# Подготовка и анализ данных о продажах видеоигр за 2000–2013 годы


### Цели и задачи проекта
Цель данного проекта - подготовить данные о продажах видеоигр для дальнейшего исследования развития игровой индустрии в период с 2000 по 2013 год. Для достижения этой цели необходимо загрузить и изучить исходный датасет, проверить качество данных, выявить и исправить возможные ошибки, обработать пропущенные значения, устранить явные и неявные дубликаты, привести названия столбцов к единому стилю и скорректировать типы данных при необходимости. После предобработки требуется отобрать данные за интересующий период, провести категоризацию игр по оценкам пользователей и критиков, выделить топ-7 игровых платформ по количеству выпущенных игр и подготовить итоговый набор данных для дальнейшего анализа развития игровой индустрии начала XXI века.

### Описание данных

Датасет new_games.csv содержит содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр

Описание столбцов:

- `Name` - название игры;
- `Platform` - игровая платформа;
- `Year of Release` - год выпуска игры;
- `Genre` - жанр игры;
- `NA sales` - продажи в Северной Америке (млн копий);
- `EU sales` - продажи в Европе (млн копий);
- `JP sales` - продажи в Японии (млн копий);
- `Other sales` - продажи в других странах (млн копий);
- `Critic Score` - оценка критиков (от 0 до 100);
- `User Score` - оценка пользователей (от 0 до 10);
- `Rating` - возрастной рейтинг ESRB.

### Содержимое проекта
1. Загрузка данных и знакомство с ними
2. Проверка ошибок в данных и их предобработка
3. Фильтрация данных
4. Категоризация данных
5. Итоговый вывод
---

## 1. Загрузка данных и знакомство с ними 


- Загрузим необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.


In [1]:
# Импортируем библиотеку pandas
import pandas as pd 

In [2]:
# Выгружаем данные из датасета new_games.csv в датафрейм game
game = pd.read_csv('/datasets/new_games.csv') 

- Познакомимся с данными: выведем общую информацию о датафрейме методом info(), а затем несколько первых строк для обзора:


In [3]:
# Выводим информацию о датафрейме
game.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [4]:
# Выводим первые строки датафрейма на экран
game.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Датасет `new_games.csv` содержит 16965 строк и 11 столбцов. Данные соответствуют описанию проекта и содержат информацию о названии игры, платформе, годе выпуска, жанре, продажах в различных регионах, оценках пользователей и критиков, а также возрастном рейтинге ESRB.

Изучим типы данных и их корректность:
- **Числовые данные с плавающей запятой (float64).** Четыре столбца представлены типом float64: `Year of Release`, `NA sales`, `Other sales`, `Critic Score`. Для столбцов с продажами и оценками критиков такой тип данных является корректным. Столбец `Year of Release` представлен типом float64 из-за наличия пропущенных значений. После обработки пропусков его можно преобразовать в целочисленный тип (int16)
- **Строковые значения (object)**. Семь столбцов имеют тип данных object: `Name`, `Platform`, `Genre`, `EU sales`, `JP sales`, `User Score`, `Rating` . 
 - Столбцы `Name`, `Platform`, `Genre` содержат текстовую информацию, поэтому тип object для них подходит.
 - Столбец `Rating` также хранит текстовые данные, но его можно рассматривать как категориальный признак. В этом случае лучше использовать тип category, чтобы улучшить производительность и оптимизировать память.
 - Столбцы `EU sales`, `JP sales` и `User Score` содержат числовые значения, однако они представлены типом object. Вероятной причиной являются строковые значения или некорректные записи. После обработки таких значений столбцы рекомендуется привести к числовому типу данных (float64)
 
Изучим наличие пропущенных значений:
Пропуски присутствуют в столбцах `Name`, `Year of Release`, `Genre`, `Critic Score`, `User Score` и `Rating`.
- Наибольшее количество пропусков содержится в столбцах `Critic Score`, `User Score` и `Rating`. Причиной может быть отсутствие пользовательских и экспертных оценок для части игр, а также отсутствие или недоступность информации о возрастном рейтинге ESRB.
- Небольшое количество пропусков содержится в столбцах `Name` и `Genre`. Такие записи требуют дополнительного изучения, так как отсутствие названия игры или жанра может затруднить дальнейший анализ.
- Пропуски в столбце `Year of Release` могут быть связаны с неполнотой исторических данных или ошибками при сборе информации.

После  анализа видно, что данные соответствуют описанию, однако требуют предобработки. Необходимо привести названия столбцов к единому стилю написания, обработать пропущенные значения, преобразовать столбцы к корректным типам и проверить датасет на наличие явных и неявных дубликатов.

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма


In [5]:
# Выводим названия всех столбцов датафрейма, чтобы проверить их стиль написания
game.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

- В названиях столбцов использовались заглавные буквы и пробелы. Для удобства дальнейшей работы все названия столбцов приведем к единому стилю - переведем в нижний регистр, а пробелы заменим на символ подчёркивания _

In [6]:
# Приводим названия столбцов к стилю snake_case
game.columns = game.columns.str.lower().str.replace(' ', '_')

# Проверяем названия всех столбцов
game.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

### 2.2. Типы данных


In [7]:
# Преобразовываем столбцы (`eu_sales`, `jp_sales`, `user_score`) с числовыми значениями к соответствующим типам данных:
game['eu_sales'] = pd.to_numeric(game['eu_sales'], errors='coerce')
game['jp_sales'] = pd.to_numeric(game['jp_sales'], errors='coerce')
game['user_score'] = pd.to_numeric(game['user_score'], errors='coerce')

In [8]:
# Преобразовываем тип данных `rating` к типу category:
game['rating'] = game['rating'].astype('category')

In [9]:
# Проверяем результат преобразования
game[['eu_sales', 'jp_sales', 'user_score', 'rating']].dtypes

eu_sales       float64
jp_sales       float64
user_score     float64
rating        category
dtype: object

Кроме приведения столбцов `eu_sales`, `jp_sales` и `user_score` к числовому типу данных, столбец `rating` был преобразован в тип category. Такое преобразование является целесообразным, поскольку данный столбец содержит ограниченный набор категориальных значений возрастного рейтинга ESRB. Использование типа category позволяет уменьшить потребление памяти и повысить эффективность работы с данными при анализе и группировке.

Столбец `year_of_release` должен содержать целочисленные значения. Однако из-за наличия пропусков его преобразование в тип int16 будет выполнено после обработки пропущенных значений.

### 2.3. Наличие пропусков в данных


In [10]:
# Выводим количество пропущенных строк в датафрейме
game.isna().sum() 

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [11]:
# Подсчитываем процент строк с пропусками
game.isna().sum() / len(game) * 100

name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating             40.522529
dtype: float64

В датасете были обнаружены пропущенные значения в 8 столбцах:

- `name` - 2 пропуска (0.01%);
- `year_of_release` - 275 пропусков (1.62%);
- `genre` - 2 пропуска (0.01%);
- `eu_sales` - 6 пропусков (0.04%);
- `jp_sales` - 4 пропуска (0.02%);
- `critic_score` - 8714 пропусков (51.39%);
- `user_score` - 9268 пропусков (54.66%);
- `rating` - 6871 пропуск (40.52%).

Наибольшее количество пропусков содержится в столбцах `critic_score`, `user_score`. Вероятно, часть игр не получила пользовательских или экспертных оценок. Замена средним значением может существенно исказить результаты анализа, поэтому пропуски лучше оставить без изменений. Столбец `rating` также содержит много пропусков, возможно часть игр могла не проходить сертификацию ESRB либо информация не была собрана. Можно заменить пропуски специальным значением-индикатором "Unknown". 

После преобразования столбцов `eu_sales` и `jp_sales` в числовой формат были обнаружены дополнительные некорректные значения. Они были автоматически заменены на пропуски (NaN), что свидетельствует о наличии ошибок или нечисловых значений в исходных данных.

Пропуски в столбцах `name`, `genre` составляют незначительную долю данных и могут быть удалены без существенного влияния на результаты анализа. Пропуски в столбце `year_of_release` можно удалить, поскольку год выпуска необходим для дальнейшей фильтрации данных по периоду 2000 - 2013 годов. Определить его без дополнительных источников невозможно.

In [12]:
# Удаляем пропуски в столбцах name, genre, year_of_release 
game = game.dropna(subset=['name', 'genre', 'year_of_release'])

In [13]:
# После удаления пропусков преобразовываем тип данных year_of_release 
game['year_of_release'] = game['year_of_release'].astype('int16')

In [14]:
# Заменяем пропуски в столбце eu_sales на среднее значение в зависимости от платформы и года выпуска игры
mean_sales = game.groupby(['platform', 'year_of_release'])['eu_sales'].mean()
def sales(row):
    if pd.isna(row['eu_sales']):
        return mean_sales[row['platform'], row['year_of_release']]
    return row['eu_sales']

game['eu_sales'] = game.apply(sales, axis=1)

In [15]:
# Заменяем пропуски в столбце jp_sales на среднее значение в зависимости от платформы и года выпуска игры
mean_sales = game.groupby(['platform', 'year_of_release'])['jp_sales'].mean()
def sales(row):
    if pd.isna(row['jp_sales']):
        return mean_sales[row['platform'], row['year_of_release']]
    return row['jp_sales']

game['jp_sales'] = game.apply(sales, axis=1)

In [16]:
# Заменяем пропуски в столбце rating на значение-индикатор "Unknown"
game['rating'] = game['rating'].cat.add_categories(['Unknown'])
game['rating'] = game['rating'].fillna('Unknown')

### 2.4. Явные и неявные дубликаты в данных

In [17]:
# Изучаем уникальные значения категориальных данных для проверки неявных дубликатов
for col in ['genre', 'platform', 'rating']:
    display(game[col].unique())

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION', 'SHOOTER',
       'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE', 'SIMULATION',
       'PUZZLE', 'STRATEGY'], dtype=object)

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

['E', 'Unknown', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Categories (9, object): ['E', 'Unknown', 'M', 'T', ..., 'K-A', 'AO', 'EC', 'RP']

In [18]:
# Приводим строки к единому формату:

# Столбец genre приводим к нижнему регистру
game['genre'] = game['genre'].str.lower()

# Столбец platform приводим к верхнему регистру
game['platform'] = game['platform'].str.upper()

# Столбец rating приводим к верхнему регистру
game['rating'] = game['rating'].astype(str).str.upper().astype('category')

# Проверяем данные после нормализации
for col in ['genre', 'platform', 'rating']:
    display(game[col].unique())

array(['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc',
       'shooter', 'simulation', 'action', 'fighting', 'adventure',
       'strategy'], dtype=object)

array(['WII', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XONE',
       'WIIU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

['E', 'UNKNOWN', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Categories (9, object): ['E', 'UNKNOWN', 'M', 'T', ..., 'K-A', 'AO', 'EC', 'RP']

- После того как нормализовали данные и устранили неявные дубликаты, проверим наличие явных дубликатов в данных.

In [19]:
# Выводим количество явных дубликатов
game.duplicated().sum()

235

In [20]:
# Удаляем явные дубликаты
game = game.drop_duplicates()

В ходе анализа было выявлено 235 явных дубликатов. Они представляют собой полностью идентичные строки и могут возникнуть, например, из-за повторной выгрузки данных.

Для корректности последующего анализа дубликаты были удалены из датасета с помощью метода drop_duplicates().После обработки каждая запись в данных стала уникальной.

In [21]:
# Количество строк в начале
initial_rows = 16956 

# Количество строк после обработки
final_rows = len(game)

# Подсчитываем количество удаленных строк в абсолютном значении
deleted_rows = initial_rows - final_rows

# Подсчитываем количество удаленных строк в относительном значении
deleted_percent = round(deleted_rows/initial_rows * 100, 2)

display(deleted_rows, deleted_percent)

512

3.02

В ходе предобработки данных были удалены строки с пропущенными значениями в столбцах `name`, `genre`, `year_of_release`. Также были устранены неявные и явные дубликаты. 

Всего было удалено 512 строк, что составляет 3.02% от исходного объема данных. После очистки датасет содержит 16 444 строк и готов для дальнейшего анализа.

---

## 3. Фильтрация данных

Фильтруем данные за период с 2000 по 2013 год включительно и сохраняем новый срез данных в отдельном датафрейме `df_actual`.

In [22]:
df_actual = game[(game['year_of_release'] >= 2000) & (game['year_of_release'] <= 2013)].copy()

---

## 4. Категоризация данных
    
Проводим категоризацию данных:
- Разделяем все игры по оценкам пользователей и выделяем такие категории: 
 - высокая оценка (от 8 до 10 включительно), 
 - средняя оценка (от 3 до 8, не включая правую границу интервала),
 - низкая оценка (от 0 до 3, не включая правую границу интервала).

In [23]:
def user_category(score):
    if pd.isna(score):
        return 'нет оценки'
    if 0 <= score <3:
        return 'низкая оценка'
    elif 3 <= score < 8:
        return 'средняя оценка'
    elif 8 <= score <=10:
        return 'высокая оценка'
df_actual['group_user_score'] = df_actual['user_score'].apply(user_category)

- Разделяем все игры по оценкам критиков и выделяем такие категории: 
 - высокая оценка (от 80 до 100 включительно), 
 - средняя оценка (от 30 до 80, не включая правую границу интервала), 
 - низкая оценка (от 0 до 30, не включая правую границу интервала).

In [24]:
def critic_category(score):
    if pd.isna(score):
        return 'нет оценки'
    if 0 <= score <30:
        return 'низкая оценка'
    elif 30 <= score < 80:
        return 'средняя оценка'
    elif 80 <= score <=100:
        return 'высокая оценка'
df_actual['group_critic_score'] = df_actual['critic_score'].apply(critic_category)

In [25]:
# Группируем данные по категориям пользовательских оценок group_user_score и оценок критиков group_critic_score
# и подсчитываем количество игр в каждой категории
df_user = df_actual['group_user_score'].value_counts()
df_critic = df_actual['group_critic_score'].value_counts()
display(df_user, df_critic)

нет оценки        6298
средняя оценка    4081
высокая оценка    2286
низкая оценка      116
Name: group_user_score, dtype: int64

нет оценки        5612
средняя оценка    5422
высокая оценка    1692
низкая оценка       55
Name: group_critic_score, dtype: int64

- Выделяем топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [26]:
df_actual['platform'].value_counts().head(7)

PS2     2127
DS      2120
WII     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: platform, dtype: int64

---

## 5. Итоговый вывод

В ходе выполнения проекта была выполнена предобработка и подготовка данных о продажах видеоигр для дальнейшего анализа в период с 2000 по 2013 год. 
Исходный датасет `new_games.csv` содержал 16 956 записей об играх и включал информацию о платформах, жанрах, годах выпуска, продажах в различных регионах мира, оценках пользователей и критиков, а также возрастном рейтинге ESRB.

В процессе предобработки данных были выполнены следующие этапы:
- названия столбцов приведены к единому стилю; 
- исправлены некорректные типы данных: числовые столбцы (`eu_sales`, `jp_sales`, `user_score`) были приведены к типу float64, столбец `rating` к типу category и столбец `Year of Release` преобразован в тип int16 для оптимизации использования памяти;
- выявлены и обработаны пропущенные значения: строки с отсутствующими значениями в ключевых столбцах (`name`, `genre`, `year_of_release`) были удалены, в столбцах `eu_sales` и `jp_sales` такие строки были заменены средними значениями по платформе и году выпуска игры, в столбце `rating` пропуски заменены на значение-индикатор Unknown;
- удалены явные дубликаты, а также приведены к единому формату текстовые значения для исключения неявных дубликатов.

В результате очистки данных было удалено 512 строк, что составляет 3,02% от исходного объёма датасета. После предобработки датасет содержит информацию о 16 444 играх.

После подготовки данных был сформирован отдельный срез `df_actual`, содержащий только игры, выпущенные в период с 2000 по 2013 год включительно. Для дальнейшего анализа были созданы новые столбцы:
- group_user_score - категория пользовательской оценки игры (низкая, средняя или высокая);
- group_critic_score - категория оценки критиков (низкая, средняя или высокая).

Также были определены наиболее популярные платформы по количеству выпущенных игр за рассматриваемый период.

Таким образом, данные были подготовлены для дальнейшего исследования тенденций игровой индустрии начала XXI века.